# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library. You will inspect record sets and fields by their Croissant `@id`, extract data, and execute basic analysis and visualization.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This can take several seconds, as the `Dataset` object performs schema validation and metadata download.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Optionally, inspect license and keywords
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs using Croissant metadata. All entities are referenced by their `@id` for consistency.

> **Note:** The FAIR² schema supports multiple record sets. We enumerate all available record set `@id`s below and then show each record set's fields.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.metadata.record_sets
print("Available Record Sets and their @id:")
for rs in record_sets:
    print(f"  - name: {rs.name}    @id: {rs.id}")

# Display fields for each record set
for rs in record_sets:
    print(f"\nRecordSet: {rs.name} (@id: {rs.id})")
    print("Fields and their @id:")
    for f in rs.fields:
        print(f"  - {f.name}   @id: {f.id}   datatype: {f.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this example, we'll extract the main data table, which contains protein identification and quantification. You can edit the variable `target_record_set_id` to extract a different table if needed (refer to IDs above).

In [ ]:
# Choose record set(s) by @id
# Here, we pick the first available record set as the primary table for demonstration.
record_sets = dataset.metadata.record_sets
record_set_ids = [rs.id for rs in record_sets]
target_record_set_id = record_set_ids[0]  # <-- replace with desired record set @id as needed

# Read all records for every record set
dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} records for record set @id: {rec_id}")

print("\nColumns in the extracted DataFrame for the main record set:")
print(dataframes[target_record_set_id].columns.tolist())
dataframes[target_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's analyze the data:
- Filter records by a numeric field using the field's `@id`
- Normalize the numeric field
- Group and aggregate by a categorical field

> To proceed, pick a numeric field and group field. These should be referenced by their field `@id`.

We auto-detect the first numeric field. You can override `numeric_field_id` and `group_field_id` to experiment with other fields.

In [ ]:
import numpy as np

df = dataframes[target_record_set_id]
# Discover numeric/categorical fields using Croissant schema
fields = [f for f in dataset.metadata.record_set(target_record_set_id).fields]
numeric_fields = [f for f in fields if f.data_type in ('Float', 'Integer', 'Number')]
categorical_fields = [f for f in fields if f.data_type in ('Text', 'String')]

if numeric_fields:
    numeric_field_id = numeric_fields[0].id
    print(f"Using numeric field (by @id): {numeric_field_id}")
else:
    numeric_field_id = df.select_dtypes(include=np.number).columns[0]
    print(f"Falling back, using first numeric DataFrame column: {numeric_field_id}")

if categorical_fields:
    group_field_id = categorical_fields[0].id
    print(f"Using group field (by @id): {group_field_id}")
else:
    group_field_id = df.select_dtypes(exclude=np.number).columns[0]
    print(f"Falling back, using first categorical DataFrame column: {group_field_id}")

threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head(3))

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

# Group and aggregate
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of a numeric field and its relationship to the group field using matplotlib and seaborn.

> Bar plots and histograms can help discover trends and anomalies.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id], kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.tight_layout()
plt.show()

# If grouping is possible, visualize means per group
if group_field_id in df.columns:
    group_means = df.groupby(group_field_id)[numeric_field_id].mean().nlargest(10)  # Top 10 groups
    plt.figure(figsize=(10, 4))
    sns.barplot(x=group_means.index, y=group_means.values)
    plt.xticks(rotation=75)
    plt.title(f"Top 10 {group_field_id} (by mean {numeric_field_id})")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, you learned to:
- Load a Croissant-based FAIR² dataset with `mlcroissant`
- Inspect available record sets, fields, and IDs (`@id`) from metadata
- Extract data into DataFrames for any record set by its `@id`
- Filter, normalize, group, and visualize records by field `@id`

This workflow supports reproducible, standards-compliant data discovery and processing across FAIR datasets.